# Neural network for Collaborative Filtering (NCF), 2017


models
- gmf
- mlp: 32 -> 16 -> 8
- neumf: gmf + mlp

exp setting
- activation: relu
- loss: log loss
- mlp init: gaussian(0, 0.01), adam
- neulml init: mlp, gmf pre-trained, sgd
- metrics: HR@10, NDCG@10
- model selection: leave-one-out
- negative sampling 

In [1]:
import os, json, bson

import numpy as np
import pandas as pd

import tensorflow as tf

2023-07-13 17:37:17.808830: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-13 17:37:17.996407: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-07-13 17:37:18.712695: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64
2023-07-13 17:37:18.712796: W tensorflow/stream_

# Data Load 

In [2]:
pinterest_dir = '../data/pinterest_iccv/'

In [3]:
f = open(os.path.join(pinterest_dir, 'subset_iccv_board_pins.bson'), 'rb')
board_pins = []
for rows in bson.decode_file_iter(f):
    board_pins.append(rows)
len(board_pins)

46000

In [4]:
user = board_pins[0]['board_id']
pinned = board_pins[0]['pins']

# Prepare Train/Test data

In [5]:
all_pins = set()

for b in board_pins:
    for p in b['pins']:
        all_pins.add(p)

all_pins = list(all_pins)
len(all_pins)

2565241

In [6]:
train_pos, test_pos, train_neg, test_neg = [], [], [], []
for b in board_pins:
    for p in b['pins'][:-1]:
        train_pos.append([b['board_id'], p])
    test_pos.append([b['board_id'],b['pins'][-1]])

    idx = 0
    negs = []
    while idx < len(b['pins']):
        randint = np.random.randint(0, 5043)
        if all_pins[randint] in b['pins']:
            continue
        else:
            negs.append([b['board_id'], all_pins[randint]])
            idx += 1
    train_neg.extend(negs[:-1])
    test_neg.append(negs[-1])

len(train_pos), len(test_pos), len(train_neg), len(test_neg)

(2519242, 46000, 2519242, 46000)

In [7]:
all_users = {tp[0] for tp in train_pos+train_neg+test_pos+test_neg}
all_items = {tp[1] for tp in train_pos+train_neg+test_pos+test_neg}

user_map = {u:i for i, u in enumerate(all_users)}
item_map = {u:i for i, u in enumerate(all_items)}

len(all_users), len(all_items)

(46000, 2565241)

In [8]:
train_X, test_X, train_y, test_y = [], [], [], []

for row in train_pos:
    train_X.append([user_map[row[0]], item_map[row[1]]])
    # train_y.append([1])
    train_y.append(1)

for row in train_neg:
    train_X.append([user_map[row[0]], item_map[row[1]]])
    # train_y.append([0])
    train_y.append(0)

for row in test_pos:
    test_X.append([user_map[row[0]], item_map[row[1]]])
    # test_y.append([1])
    test_y.append(1)

for row in test_neg:
    test_X.append([user_map[row[0]], item_map[row[1]]])
    # test_y.append([0])
    test_y.append(0)

train_X = np.array(train_X, dtype='float64')
test_X = np.array(test_X, dtype='float64')

In [9]:
def onehot(index, depth):
    vector = np.zeros(depth)
    print(index[0])
    print(depth)
    vector[int(index[0])] = 1
    return vector


In [11]:
train_X_user = train_X[:, 0]
train_X_item = train_X[:, 1]
test_X_user = test_X[:, 0]
test_X_item = test_X[:, 1]

# Models

In [13]:
class NCF(tf.keras.Model):

    def __init__(self):
        pass

In [14]:
# gmf
class GMF(NCF):
    def __call__(self):
        pass

# gmf
class MLP(NCF):
    def __call__(self):
        pass

# gmf
class NeuMF(NCF):
    def __call__(self):
        pass

In [7]:
def get_gmf(user_input_size, item_input_size, embedding_size):
    
    user_input = tf.keras.layers.Input(shape=user_input_size, dtype="float32")
    item_input = tf.keras.layers.Input(shape=item_input_size, dtype="float32")

    user_encoding = tf.keras.layers.CategoryEncoding(num_tokens=46000, output_mode="one_hot")(user_input)
    item_encoding = tf.keras.layers.CategoryEncoding(num_tokens=2565241, output_mode="one_hot")(item_input)

    user_embedding = tf.keras.layers.Dense(units=embedding_size, dtype="float32", 
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(user_encoding)
    item_embedding = tf.keras.layers.Dense(units=embedding_size, dtype="float32", 
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(item_encoding)

    x = tf.keras.layers.Concatenate()([user_embedding, item_embedding])
    x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32",
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    model = tf.keras.Model([user_input, item_input], x)
    return model

In [16]:
# def get_mlp(user_input_size, item_input_size, embedding_size, layer_units, predictive_factor_size):
    
#     user_input = tf.keras.layers.Input(shape=user_input_size, dtype="float32")
#     item_input = tf.keras.layers.Input(shape=item_input_size, dtype="float32")

#     user_encoding = tf.keras.layers.CategoryEncoding(num_tokens=46000, output_mode="one_hot")(user_input)
#     item_encoding = tf.keras.layers.CategoryEncoding(num_tokens=2565241, output_mode="one_hot")(item_input)

#     user_embedding = tf.keras.layers.Dense(units=embedding_size, dtype="float32", 
#         kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(user_encoding)
#     item_embedding = tf.keras.layers.Dense(units=embedding_size, dtype="float32", 
#         kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(item_encoding)

#     x = tf.keras.layers.Concatenate()([user_embedding, item_embedding])

#     for l in layer_units:
#         x = tf.keras.layers.Dense(units=l, activation="relu", dtype="float32",
#             kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)
#     x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32",
#         kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

#     model = tf.keras.Model([user_input, item_input], x)
#     return model

In [8]:
gmf_model = get_gmf(1, 1, 16)
gmf_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 category_encoding (CategoryEnc  (None, 46000)       0           ['input_1[0][0]']                
 oding)                                                                                           
                                                                                                  
 category_encoding_1 (CategoryE  (None, 2565241)     0           ['input_2[0][0]']            

2023-07-13 17:41:03.553106: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-13 17:41:03.556593: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-13 17:41:03.569255: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-13 17:41:03.572648: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-13 17:41:03.575907: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from S

Total params: 41,779,921
Trainable params: 41,779,921
Non-trainable params: 0
__________________________________________________________________________________________________


In [18]:
# mlp_model = get_mlp(100, 5043, 16, [32, 16], 8)
# mlp_model.summary()

In [19]:
learning_rate = 0.00001
gmf_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="binary_crossentropy", metrics=['acc'])

In [20]:
# gmf_model.fit([train_X[:, 0], train_X[:, 1]], train_y)

In [21]:
np.expand_dims(np.array(train_y, dtype='float32'), axis=1).shape

(5038484, 1)

In [22]:
train_X_user.shape

(5038484,)

In [23]:
gmf_model.fit([train_X_user, train_X_item], np.array(train_y, dtype='float32'), 
    epochs=10, batch_size=125)

Epoch 1/10
40308/40308 [==============================] - 1226s 30ms/step - loss: 0.6670 - acc: 0.9368
Epoch 2/10
40308/40308 [==============================] - 1220s 30ms/step - loss: 0.5332 - acc: 0.9990
Epoch 3/10
40308/40308 [==============================] - 1221s 30ms/step - loss: 0.3357 - acc: 0.9990
Epoch 4/10
40308/40308 [==============================] - 1220s 30ms/step - loss: 0.1618 - acc: 0.9990
Epoch 5/10
40308/40308 [==============================] - 1221s 30ms/step - loss: 0.0608 - acc: 0.9990
Epoch 6/10
40308/40308 [==============================] - 1221s 30ms/step - loss: 0.0206 - acc: 0.9990
Epoch 7/10
40308/40308 [==============================] - 1220s 30ms/step - loss: 0.0095 - acc: 0.9990
Epoch 8/10
40308/40308 [==============================] - 1220s 30ms/step - loss: 0.0074 - acc: 0.9990
Epoch 9/10
40308/40308 [==============================] - 1220s 30ms/step - loss: 0.0070 - acc: 0.9990
Epoch 10/10
40308/40308 [==============================] - 1219s 30ms/ste

In [24]:
train_X_user.shape, train_X_item.shape, len(train_y)

((5038484,), (5038484,), 5038484)

In [29]:
test_pred = gmf_model.predict([test_X_user, test_X_item])

2875/2875 [==============================] - 15s 5ms/step


In [27]:
from sklearn.metrics import accuracy_score

In [38]:
test_pred_2 = test_pred >= 0.5

In [39]:
accuracy_score(test_y, test_pred_2)

0.9989130434782608

In [40]:
from sklearn.metrics import ndcg_score

In [42]:
user_scores = {u:[] for u in all_users}
for u in zip(test_X_user, test_y):
    user_scores[u].append()

array([ 1005.,  2527., 43809., ..., 43994., 23269., 28389.])

In [ ]:
ndcg_score

---

In [ ]:
@tf.function
def hit_ratio(y_true, y_pred):
    y_pred = [int(x >= 0.5) for x in y_pred]
    hits = [int(x==y==1) for x,y in zip(y_pred, y_true)]
    return sum(hits)/sum(y_pred)

In [ ]:
hit_ratio([1, 0, 1], [0.2, 0.1, 0.9])

<tf.Tensor: shape=(), dtype=float32, numpy=1.0>

In [ ]:
import tensorflow as tf
tf.reduce_sum([0,0,1])

<tf.Tensor: shape=(), dtype=int32, numpy=1>

In [ ]:
# class hit_ratio(tf.keras.metrics.Metric):
#     def __init__(self, name = 'hit_ratio', **kwargs):
#         super(hit_ratio, self).__init__(**kwargs)
#         self.hits = None

#     def update_state(self, y_true, y_pred, sample_weight=None):
#         y_true = tf.cast(y_true, tf.bool)
#         y_pred = tf.math.greater(y_pred, 0.5)
#         self.hits = tf.logical_and(tf.equal(y_true, y_pred), tf.equal(y_true, 1))
#         self.targets = tf.reduce_sum(y_pred)
        
#     def reset_state(self):
#         self.hits.assign(0)

#     def result(self):
#         return tf.reduced_sum(self.hits)/sum(y_pred)

In [ ]:
# def hr_keras(y_true, y_pred):
#     score = tf.py_function(func=hit_ratio, inp=[y_true, y_pred], Tout=tf.float64,  name='custom_hr') # tf 2.x
#     #score = tf.py_func( lambda y_true, y_pred : mse_AIFrenz(y_true, y_pred) , [y_true, y_pred], 'float32', stateful = False, name = 'custom_mse' ) # tf 1.x
#     return score